# Figure 1 E — d2 by state on non-overlapping windows

Each panel is saved to `reports/revision_final/<Fig>/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))   # notebooks/figures_revision
import final_figs as F                     # paths, save/copy helpers; adds the revision folder to sys.path

## Fig 1E — d2 by state, non-overlapping windows

d2 (2024 Euclidean, AR p = 20, 40 ms, full window) of the non-overlapping windows (per session, centres ≥ 30 s apart; 13 536 windows, 18 sessions), drawn by the Fig 1d notebook of the first submission (`Previous_Rv1/figure_scripts/Fig1_D_histogram_S1_E.ipynb`). Per state a KDE (bw_adjust 0.5) with a box-plot inset.

- Sleep = NREM 6 040 + REM 227; Wake = θ 1 602 + non-θ 5 667, θ / non-θ by the state at the window centre.
- Inset p: session-paired signed-rank test of session medians.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

TABLE = os.path.join(F.REPO, "data", "revision", "log", "fig1e_d2_windows_disjoint.csv")
if not os.path.exists(TABLE):            # the exact table the submitted notebook plots, on disjoint windows
    F.rv1_table("Fig1_D_histogram_S1_E", 'df_30s_only = df_binned[df_binned["window"] == "30s"].copy()',
                "df_30s_only", TABLE)
T = pd.read_csv(TABLE)
# d2 (2024 Euclidean)
T["session"] = T.animal.astype(str) + "_day" + T.day.astype(str)
# (title, rows, states in box order, colours, tick labels)
# Wake = every wake window, theta / non-theta by the state at the window centre (the paper's label).
ARMS = [("Wake", T[T.state.isin(["THETA", "nonTHETA"])], ["THETA", "nonTHETA"],
         {"THETA": "#d95f0e", "nonTHETA": "#fdb863"}, ["θ", "non-θ"]),
        ("Sleep", T[T.state.isin(["NREM", "REM"])], ["NREM", "REM"],
         {"NREM": "#7b3294", "REM": "#d7b5d8"}, ["NREM", "REM"])]


def paired_sessions(d, a, b):
    """Session medians of d2 in states a and b, paired over sessions with both; Wilcoxon signed-rank."""
    m = d.groupby(["session", "state"]).d2.median().unstack().dropna(subset=[a, b])
    diff = m[a] - m[b]
    return dict(n=len(m), n_pos=int((diff > 0).sum()), median_diff=float(diff.median()),
                p=float(stats.wilcoxon(m[a], m[b]).pvalue))


F.final_style()
fig, axes = plt.subplots(1, 2, figsize=(74 * F.MM, 42.5 * F.MM), constrained_layout=True)
rows = []
for ax, (title, d, order, pal, ticks) in zip(axes, ARMS):
    sns.kdeplot(data=d, x="d2", hue="state", hue_order=order[::-1], palette=pal, fill=True, alpha=0.2,
                common_norm=False, bw_adjust=0.5, linewidth=0.9, legend=False, ax=ax)
    ax.set_xlim(5e-3, 2e-1); ax.set_xticks([0.05, 0.1, 0.15, 0.2]); ax.set_xticklabels(["0.05", "0.10", "0.15", "0.20"])
    ax.set_xlabel("d2"); ax.set_ylabel("Probability density" if ax is axes[0] else "")
    ax.set_title(title, loc="left")
    ins = ax.inset_axes([0.6, 0.38, 0.38, 0.6])
    sns.boxplot(data=d, x="state", y="d2", order=order, hue="state", hue_order=order, palette=pal, legend=False,
                showfliers=False, width=0.6, linewidth=0.5, ax=ins)
    ins.set_xticks([0, 1]); ins.set_xticklabels(ticks, fontsize=5); ins.set_xlabel(""); ins.set_ylabel("d2", labelpad=1)
    q = [np.percentile(d.loc[d.state == k, "d2"], [25, 75]) for k in order]
    yb = 1.06 * max(d.loc[(d.state == k) & (d.d2 <= q3 + 1.5 * (q3 - q1)), "d2"].max()   # above the upper whiskers
                    for k, (q1, q3) in zip(order, q))
    ins.set_ylim(0, yb * 1.2); ins.set_yticks([t for t in (0, 0.1, 0.2) if t <= yb * 1.2]); ins.set_yticklabels([f"{t:g}" for t in ins.get_yticks()])
    ins.tick_params(labelsize=5, length=1.5, pad=1)
    s = paired_sessions(d, order[0], order[1])
    rows.append(dict(arm=title, contrast=f"{order[0]} - {order[1]}", n_windows=len(d),
                     **{f"n_{k}": int((d.state == k).sum()) for k in order}, **s))
    ins.plot([0, 0, 1, 1], [yb * 0.97, yb, yb, yb * 0.97], color="k", lw=0.5)
    ins.text(0.5, yb * 1.02, F.stars(s["p"]), ha="center", va="bottom", fontsize=6)   # session-paired test
S = pd.DataFrame(rows)
print(S[["arm", "contrast", "n", "p"]].rename(columns={"n": "n_sessions", "p": "p (two-sided paired Wilcoxon)"}).to_string(index=False))
png = F.save(fig, "Fig1_E_d2_histogram_nonoverlap")
F.show(png)